In [4]:
import pandas as pd
import gspread
import gspread_dataframe as gd
import re
import os
from oauth2client.service_account import ServiceAccountCredentials
from datetime import datetime
from dataclasses import dataclass
from typing import Dict

WAREHOUSE_SHEET_ID = '1AqNlegU_wTy5HpEcf1ThwOfe8K_81rGoPIvByGPK6RI'

@dataclass
class GoogleSheet:
    id: str
    pages: Dict[str, str]

GOOGLE_SHEETS = {
    "Raport_incasari" : GoogleSheet(
        id="1Cj6mNa6kaPOwckd-REVZsemOyCWKRPuMMIaVlwISxLc",
        pages={
            "Incasari_Iunie_2025": "1455833454",
            "Incasari_Mai_2025" : "1829298788",
            "Incasari_Aprilie_2025" : "1936663589",
            "Incasari_Martie_2025" : "1026830607",
            "Incasari_Februarie_2025" : "881321237",
            "Incasari_Ianuarie_2025" : "1302372194"
        }
    )
}

KNOWN_BRANDS = [
    'ANA HICKMANN', 'ARMANI EXCHANGE', 'ARNETTE', 'AVANGLION', 'BABY-HIPPO', 
    'BURBERRY', 'BVLGARI', 'CALVIN KLEIN', 'CELINE', 'DIOR', 'DOLCE&GABBANA', 
    'EMPORIO ARMANI', 'ENOX', 'ESCHENBACH', 'FIELMANN', 'FITCHE', 'FURLA', 'GANT', 
    'GIORGIO ARMANI', 'GUESS', 'HARLEY DAVIDSON', 'HELLO KITTY', 'HICKMANN', 
    'JIMMY CHOO', 'MICHAEL KORS', 'MISS SIXTY', 'OAKLEY', 'POLICE', 'PRADA', 
    'RALPH LAUREN', 'RAY BAN', 'SFEROFLEX', 'SILHOUETTE', 'SWAROVSKI', 'VALENTINO', 
    'VERSACE', 'VOGUE', 'CHARM', 'PREMIUM', 'MIEN', 'TRIQ', 'SUCCESS', 'POLAR', 
    'AVIZOR', 'CORSO', 'LUNETTES', 'LEVEL', 'OPT', 'HUGO BOSS', 'TOM FORD'
]

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)

In [5]:
def get_gspread_client(json_path='../credentials.json'):
    if not os.path.exists(json_path):
        json_path = 'credentials.json' 
    if not os.path.exists(json_path):
        raise FileNotFoundError(f"Nu găsesc fișierul de credențiale la: {json_path}")
    scope = ["https://spreadsheets.google.com/feeds", "https://www.googleapis.com/auth/drive"]
    creds = ServiceAccountCredentials.from_json_keyfile_name(json_path, scope)
    return gspread.authorize(creds)

def load_sheet_data(client, sheet_config, page_name, header_row_index=3):
    try:
        spreadsheet = client.open_by_key(sheet_config.id)
        gid_target = sheet_config.pages.get(page_name)
        if not gid_target or gid_target == "id":
            return None 
            
        worksheet = None
        for ws in spreadsheet.worksheets():
            if str(ws.id) == str(gid_target):
                worksheet = ws
                break
        if not worksheet:
            print(f"Worksheet GID {gid_target} not found.")
            return None

        print(f"Downloading data from: {page_name}...")
        all_values = worksheet.get_all_values()
        
        if len(all_values) <= header_row_index:
            return None
            
        headers = all_values[header_row_index]
        data_rows = all_values[header_row_index + 1:]
        return pd.DataFrame(data_rows, columns=headers)

    except Exception as e:
        print(f"❌ Eroare la {page_name}: {e}")
        return None

def push_data_to_looker_source(dataframe, sheet_id, worksheet_name, client):
    try:
        sh = client.open_by_key(sheet_id)
        try:
            worksheet = sh.worksheet(worksheet_name)
        except:
            print(f"Foaia '{worksheet_name}' nu există. O creez acum.")
            worksheet = sh.add_worksheet(title=worksheet_name, rows="1000", cols="20")
            
        worksheet.clear()
        gd.set_with_dataframe(worksheet, dataframe, resize=True)
        print(f"Success! Data uploaded to the cloud.")
    except Exception as e:
        print(f"Upload error: {e}")


def clean_money(val_str):
    if not val_str: return 0.0
    clean = str(val_str).lower().replace('lei', '').replace('.', '').replace(',', '.').strip()
    try: return float(clean)
    except: return 0.0

def process_complex_report(raw_df):
    processed_rows = []
    current_bon = {'Nr_Bon': None, 'Data': None, 'Pacient': None, 'Mod_Plata': None}
    
    regex_data = r"Data:\s*(\d{4}-\d{2}-\d{2})"
    regex_pacient = r"Pacient:\s*(.*?);"
    regex_comanda = r"Bon comanda:\s*(.*?);"

    for idx, row in raw_df.iterrows():
        raw_text = str(row.get('Gestiune', ''))
        
        if "Bon comanda" in raw_text:
            match_data = re.search(regex_data, raw_text)
            match_pac = re.search(regex_pacient, raw_text)
            match_cmd = re.search(regex_comanda, raw_text)
            
            current_bon['Data'] = match_data.group(1) if match_data else None
            current_bon['Pacient'] = match_pac.group(1).strip() if match_pac else "Necunoscut"
            current_bon['Nr_Bon'] = match_cmd.group(1).strip() if match_cmd else None
            
            if "Numerar" in raw_text: current_bon['Mod_Plata'] = "Numerar"
            elif "Card" in raw_text: current_bon['Mod_Plata'] = "Card"
            elif "Ordin" in raw_text: current_bon['Mod_Plata'] = "OP"
            else: current_bon['Mod_Plata'] = "Nespecificat"
            continue

        produs_nume = str(row.get('Produs', '')).strip()
        valoare_raw = row.get('Valoare cu TVA', '')
        
        if produs_nume and produs_nume != "Produs" and valoare_raw:
            valoare = clean_money(valoare_raw)
            try: cantitate = float(str(row.get('Cantitate', '0')).replace(',', '.'))
            except: cantitate = 0

            new_row = {
                'Data_Incasare': current_bon['Data'],
                'Locatie': row.get('Gestiune', ''),
                'Nr_Bon': current_bon['Nr_Bon'],
                'Pacient': current_bon['Pacient'],
                'Produs': produs_nume,
                'Cantitate': cantitate,
                'Valoare_Totala': valoare,
                'Mod_Plata': current_bon['Mod_Plata']
            }
            processed_rows.append(new_row)
    return pd.DataFrame(processed_rows)

def get_brand_v3(text):
    text = text.upper()
    for brand in KNOWN_BRANDS:
        if brand in text: return brand
    return "GENERIC"

def get_category_v3(text, brand_found):
    text = text.upper()
    if any(x in text for x in ["MANOPERA", "MONTAJ", "CONSULT", "DATARE"]): return "Servicii"
    if any(x in text for x in [" ML", "CPS", "HYLO", "COLINERV", "SOLUTIE", "SPRAY"]): return "Tratamente & Solutii"
    if any(x in text for x in ["CONTACT", "DAILY", "ACUVUE", "OASYS"]): return "Lentile Contact"
    if any(x in text for x in ["TOC ", "LAVETE", "SNUR"]): return "Accesorii"
    if any(x in text for x in ["SV ", "PC ", "LENTILE", "1.5", "1.6", "HMC", "CAMBER", "VARILUX"]): return "Lentile Aeriene"
    if brand_found != "GENERIC": return "Rame"
    if re.search(r'[A-Z]{2,}\s*\d+', text): return "Rame (Generic)"
    return "Altele"

In [6]:
client = get_gspread_client()

all_data_frames = []
raport_config = GOOGLE_SHEETS["Raport_incasari"]

HEADER_INDEX = 2 

for month_name, gid in raport_config.pages.items():
    raw_df = load_sheet_data(client, raport_config, month_name, header_row_index=HEADER_INDEX)
    
    if raw_df is not None:
        clean_df = process_complex_report(raw_df)
        all_data_frames.append(clean_df)
        print(f"   -> {len(clean_df)} rânduri procesate.")

if all_data_frames:
    df_final = pd.concat(all_data_frames, ignore_index=True)
    df_final['Data_Incasare'] = pd.to_datetime(df_final['Data_Incasare'])
    
    print("Aplic clasificare Brand & Categorie...")
    df_final['Brand'] = df_final['Produs'].apply(get_brand_v3)
    df_final['Categorie'] = df_final.apply(lambda row: get_category_v3(row['Produs'], row['Brand']), axis=1)
    
    df_export = df_final.fillna("")
    df_export['Data_Incasare'] = df_export['Data_Incasare'].astype(str)
    
    print(df_export['Categorie'].value_counts())
    
    push_data_to_looker_source(df_export, WAREHOUSE_SHEET_ID, 'Vanzari_Consolidate', client)
    
else:
    print("No data to process.")

   -> 5392 rânduri procesate.
   -> 6635 rânduri procesate.
   -> 5630 rânduri procesate.
   -> 7169 rânduri procesate.
   -> 6109 rânduri procesate.
   -> 6501 rânduri procesate.
Aplic clasificare Brand & Categorie...
Categorie
Lentile Aeriene         7451
Servicii                6896
Rame                    6457
Rame (Generic)          5625
Accesorii               4068
Tratamente & Solutii    3758
Altele                  2980
Lentile Contact          201
Name: count, dtype: int64
Success! Data uploaded to the cloud.
